# Build query-mode inputs for sv-evidence-extraction

Computes the four inputs that vary per candidate event (`region`,
`sample_ids`, `region_name`, `output_prefix`) from a chrom/start/end and
a child `IndividualID`, resolving parent sample IDs from the cohort's
pedigree file. Merges them with the stable defaults (evidence-paths
table, sample/batch map, docker image, padding) from
[`inputs/query.inputs.json`](../inputs/query.inputs.json) to produce a
ready-to-submit Terra input JSON, and optionally uploads it straight to
the workspace bucket.

Workspace-specific paths (bucket ID, cohort filenames) live in
`local_config.json` at the repo root, which is gitignored -- see
`local_config.example.json` for the expected shape. Nothing
cohort-specific is hardcoded in this notebook itself.


In [2]:
# ==========================================
# IMPORTS
# ==========================================
import json
from pathlib import Path

import pandas as pd
from gcs_utils import upload_to_workspace_bucket


In [3]:
# ==========================================
# LOCAL CONFIG (workspace-specific paths)
# ==========================================
# local_config.json holds the real, workspace-specific GCS paths for this
# cohort (evidence-paths table, sample/batch map, pedigree file) and is
# gitignored -- see local_config.example.json for the expected shape.
# Keeping it out of the repo means no cohort-specific bucket ID or
# filenames end up in the public history, while this notebook still runs
# against real data locally.
#
# Assumes the notebook is run from its default location (notebooks/);
# adjust REPO_ROOT if you've moved it.
REPO_ROOT = Path.cwd().parent
LOCAL_CONFIG_PATH = REPO_ROOT / "local_config.json"

with open(LOCAL_CONFIG_PATH) as fh:
    local_config = json.load(fh)

local_config


{'evidence_paths_tsv': 'gs://fc-secure-634cb4b1-313c-461f-b753-d647c49c9961/uploads/SV_evidence_extraction/ASC_evidence_paths.tsv',
 'sample_batch_map_tsv': 'gs://fc-secure-634cb4b1-313c-461f-b753-d647c49c9961/uploads/sample_set_table/asd_cohort-sample_map-to_keep.tsv',
 'ped_file_uri': 'gs://fc-secure-634cb4b1-313c-461f-b753-d647c49c9961/uploads/SV_evidence_extraction/ped_ASC_WGS_basicQC_sex_check_relatednessV5_2025-08-29_forGATKSV.txt',
 'workspace_bucket': 'fc-secure-634cb4b1-313c-461f-b753-d647c49c9961',
 'upload_prefix': 'uploads/SV_evidence_extraction',
 'gcp_billing_project': 'talkowski-sv-gnomad',
 'local_evidence_dir': '/Users/murphyda/My Drive/SV_manual_plot_review/SV_evidence_extraction'}

In [4]:
# ==========================================
# PEDIGREE LOOKUP
# ==========================================
# Shared with examine_evidence.ipynb via pedigree_utils.py -- see that
# module for the "0" = no-parent PED sentinel handling.
from pedigree_utils import load_pedigree, resolve_family_sample_ids

df_ped = load_pedigree(local_config["ped_file_uri"])
df_ped.head()


/opt/anaconda3/lib/python3.13/site-packages/google/auth/_default.py:113: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


,FamID,IndividualID,FatherID,MotherID,Gender,Affected
0,11000,__ssc02217__0cb03e,0,0,2,1.0
1,11000,__ssc02219__4ee3b0,0,0,1,1.0
2,11000,__ssc02220__b84587,__ssc02219__4ee3b0,__ssc02217__0cb03e,2,1.0
3,11000,__ssc02254__21f98a,__ssc02219__4ee3b0,__ssc02217__0cb03e,1,2.0
4,11001,__ss0013024__920359,__ssc02184__a8235a,__ssc02181__ddf9cb,1,2.0


In [5]:
# ==========================================
# BUILD THE VARIABLE QUERY-MODE INPUTS
# ==========================================
def build_query_inputs(chrom, start, end, child_id, df_ped, include="both", name=None):
    """Compute the four variable sv-evidence-extraction query-mode inputs for one candidate event.

    The other query-mode inputs (evidence_paths_tsv, sample_batch_map_tsv,
    docker, padding, etc.) are stable across events and come from the
    checked-in inputs/query.inputs.json template plus local_config.json,
    not from this function -- see `write_query_inputs_json`.

    Parameters
    ----------
    chrom : str
    start, end : int
        1-based inclusive core event coordinates (padding is applied by
        the WDL/CLI itself, not here).
    child_id : str
        IndividualID of the proband, used to resolve parents via df_ped.
    df_ped : pandas.DataFrame
        As returned by `load_pedigree`.
    include : {"both", "father", "mother", "none"}, default "both"
        Which parent(s) to pull evidence for alongside the child.
    name : str, optional
        Human-readable label for this event; auto-generated from
        chrom/start/end/child_id if not given.

    Returns
    -------
    dict
        Keys "region", "sample_ids", "region_name", "output_prefix" --
        exactly the TBD fields in inputs/query.inputs.json.
    """
    sample_ids = resolve_family_sample_ids(child_id, df_ped, include=include)
    region_name = name or f"{chrom}_{start}_{end}_{child_id}"

    return {
        "region": f"{chrom}:{start}-{end}",
        "sample_ids": ",".join(sample_ids),
        "region_name": region_name,
        "output_prefix": region_name,
    }


In [6]:
# ==========================================
# MERGE INTO THE QUERY-MODE INPUT TEMPLATE
# ==========================================
def write_query_inputs_json(variable_inputs, local_config, template_path, out_path):
    """Merge computed variable inputs with the stable template/local-config defaults, and write the result.

    Parameters
    ----------
    variable_inputs : dict
        As returned by `build_query_inputs` -- region, sample_ids,
        region_name, output_prefix.
    local_config : dict
        As loaded from local_config.json -- supplies evidence_paths_tsv,
        sample_batch_map_tsv, and ped_file_uri.
    template_path : str or pathlib.Path
        Path to the checked-in inputs/query.inputs.json (structural
        template: key names, padding defaults, docker image).
    out_path : str or pathlib.Path
        Where to write the merged, ready-to-submit JSON.

    Returns
    -------
    dict
        The merged input dictionary that was written to `out_path`.
    """
    with open(template_path) as fh:
        merged = json.load(fh)

    merged["SVEvidenceExtraction.evidence_paths_tsv"] = local_config["evidence_paths_tsv"]
    merged["SVEvidenceExtraction.sample_batch_map_tsv"] = local_config["sample_batch_map_tsv"]
    merged["SVEvidenceExtraction.ped_file"] = local_config["ped_file_uri"]
    merged["SVEvidenceExtraction.region"] = variable_inputs["region"]
    merged["SVEvidenceExtraction.sample_ids"] = variable_inputs["sample_ids"]
    merged["SVEvidenceExtraction.region_name"] = variable_inputs["region_name"]
    merged["SVEvidenceExtraction.output_prefix"] = variable_inputs["output_prefix"]

    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with open(out_path, "w") as fh:
        json.dump(merged, fh, indent=2)

    return merged


## Example: one candidate event

Replace the coordinates and `child_id` below with a real candidate --
this example uses placeholder values only.


In [7]:
# ==========================================
# EXAMPLE: ONE CANDIDATE EVENT
# ==========================================
# Parses an IGV-review-style filename (matching the original analysis
# notebook's convention) to pull out the region and child_id directly,
# rather than typing coordinates by hand.
image_file = "asd_cohort.chr20.final_cleanup_DEL_chr20_8601~~__2_1738_003_recal__661056~~chr20_38104076-38108227.png"
region = image_file.replace(".png", "").split("~~")[2]
chrom = region.split("_")[0]
startend = region.split("_")[1]
start, end = startend.split("-")
child_id = image_file.replace(".png", "").split("~~")[1]

example_inputs = build_query_inputs(
    chrom=chrom,
    start=int(start),
    end=int(end),
    child_id=child_id,
    df_ped=df_ped,
    include="both",
)
example_inputs


{'region': 'chr20:38104076-38108227',
 'sample_ids': '__2_1738_003_recal__661056,__2_1738_002_recal__b3dabd,__2_1738_001_recal__07e559',
 'region_name': 'chr20_38104076_38108227___2_1738_003_recal__661056',
 'output_prefix': 'chr20_38104076_38108227___2_1738_003_recal__661056'}

In [8]:
# ==========================================
# WRITE + (OPTIONALLY) UPLOAD THE READY-TO-SUBMIT JSON
# ==========================================
out_path = REPO_ROOT / "notebooks" / "generated" / f"{example_inputs['region_name']}.query.inputs.json"
merged = write_query_inputs_json(
    variable_inputs=example_inputs,
    local_config=local_config,
    template_path=REPO_ROOT / "inputs" / "query.inputs.json",
    out_path=out_path,
)
print(f"Wrote {out_path}")

upload_to_workspace_bucket(
    local_path=out_path,
    workspace_bucket=local_config["workspace_bucket"],
    upload_prefix=local_config["upload_prefix"],
    gcp_billing_project=local_config["gcp_billing_project"],
)


Wrote /Users/murphyda/My Drive/python_code/sv-evidence-extraction/notebooks/generated/chr20_38104076_38108227___2_1738_003_recal__661056.query.inputs.json


/opt/anaconda3/lib/python3.13/site-packages/google/auth/_default.py:113: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)
I0915 16:25:22.887329 6994487 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers
I0915 16:25:22.891313 7070761 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(84, generation: 1)


Uploaded to gs://fc-secure-634cb4b1-313c-461f-b753-d647c49c9961/uploads/SV_evidence_extraction/chr20_38104076_38108227___2_1738_003_recal__661056.query.inputs.json


'gs://fc-secure-634cb4b1-313c-461f-b753-d647c49c9961/uploads/SV_evidence_extraction/chr20_38104076_38108227___2_1738_003_recal__661056.query.inputs.json'